1.1 Datenimport – PDF-Text extrahieren


In [ ]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

# Beispiel: Lade die Bilanz-PDF und gebe den extrahierten Text aus
pdf_text = extract_text_from_pdf("mercedes-benz-geschaeftsbericht-2024-inkl-zusammengefasster-lagebericht-mbg-ag.pdf")
print(pdf_text)


1.2 Texterkennung / NLP – Entitäten extrahieren


In [ ]:
from transformers import pipeline

# Beispiel: Verwende ein für deutsche Texte geeignetes Modell
nlp = pipeline("ner", model="dbmdz/bert-base-german-cased", aggregation_strategy="simple")

def extract_entities(text):
    entities = nlp(text)
    return entities

# Beispiel: Extrahiere Entitäten aus dem PDF-Text
entities = extract_entities(pdf_text)
print(entities)


1.3 Mapping in die Strukturbilanz – Zuordnung und Summierung


In [ ]:
def map_entities_to_structure(entities):
    # Initialisiere die Strukturbilanz-Kategorien
    strukturbilanz = {
        "Anlagevermögen": 0.0,
        "Umlaufvermögen": 0.0,
        "Eigenkapital": 0.0,
        "Fremdkapital": 0.0
    }
    
    for entity in entities:
        # Beispiel: Wir gehen davon aus, dass 'entity["word"]' den erkannten Text enthält.
        entity_text = entity["word"]
        
        # Versuche, einen Zahlenwert aus dem erkannten Text zu extrahieren.
        try:
            # Ersetze Punkt als Tausendertrennzeichen und Komma als Dezimaltrennzeichen
            value = float(entity_text.replace(".", "").replace(",", "."))
        except ValueError:
            continue  # Überspringe, falls kein Zahlenwert vorliegt
        
        # Einfache Logik zur Zuordnung anhand von Schlüsselwörtern.
        # (Je nach Datenlage ist eine feinere Abstimmung nötig!)
        if "Anlage" in entity_text:
            strukturbilanz["Anlagevermögen"] += value
        elif "Umlauf" in entity_text:
            strukturbilanz["Umlaufvermögen"] += value
        elif "Eigen" in entity_text:
            strukturbilanz["Eigenkapital"] += value
        elif "Fremd" in entity_text:
            strukturbilanz["Fremdkapital"] += value
    
    return strukturbilanz

# Beispiel: Mappe die erkannten Entitäten in die Strukturbilanz
strukturbilanz = map_entities_to_structure(entities)
print(strukturbilanz)
